# Prompt caching: cut latency and cost on repeated context

**The Agent Build Log · Build #006 · GenAI & LLM Foundations 6/8**

[Build #005](../05-context-windows-and-cost/) showed that every request re-sends its context. When that context is the
same every time (a big system prompt, a catalog, a policy doc), **prompt caching** lets the API reuse it instead of
re-reading it. This build measures it on Claude Haiku 4.5, including the two ways it silently fails.

1. [The setup: an 8K-token catalog prompt](#1-the-setup-an-8k-token-catalog-prompt)
2. [Silent failure #1: too short to cache](#2-silent-failure-1-too-short-to-cache)
3. [Caching done right: 10 questions, cached vs not](#3-caching-done-right-10-questions-cached-vs-not)
4. [Silent failure #2: one timestamp breaks it](#4-silent-failure-2-one-timestamp-breaks-it)
5. [When does caching pay off?](#5-when-does-caching-pay-off)
6. [Findings](#6-findings)

**Needs:** an Anthropic API key in a `.env` file. A full run costs roughly **$0.50** on Claude Haiku 4.5.

**Pricing used** (Claude Haiku 4.5, per million tokens): input $1.00 · cache write (5-minute TTL) $1.25 · cache read $0.10 · output $5.00.

In [1]:
import json, os, random, statistics, time, uuid
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

MODEL = "claude-haiku-4-5"
PRICE = {"input": 1.00, "cache_write": 1.25, "cache_read": 0.10, "output": 5.00}   # $ per 1M tokens
MIN_CACHEABLE = 4096   # Claude Haiku 4.5's minimum cacheable prefix (tokens)
client = anthropic.Anthropic()

def cost(u) -> float:
    return (u.input_tokens * PRICE["input"] + (u.cache_creation_input_tokens or 0) * PRICE["cache_write"]
            + (u.cache_read_input_tokens or 0) * PRICE["cache_read"] + u.output_tokens * PRICE["output"]) / 1e6

RUN = uuid.uuid4().hex[:8]   # makes this run's cache entries fresh, so the first cached call is always a write
print("model:", MODEL, "| run:", RUN)

model: claude-haiku-4-5 | run: 745cc1fe


## 1. The setup: an 8K-token catalog prompt

A support assistant that answers questions about a product catalog. The catalog is generated from a fixed seed,
so every run (and every reader) gets the same 150 products and the same policies.

In [2]:
KINDS = ["Wireless Mouse", "Mechanical Keyboard", "USB-C Hub", "Laptop Stand", "Webcam", "Noise-Cancelling Headset",
         "Portable SSD", "Monitor Arm", "Docking Station", "Ergonomic Chair Cushion"]
BRANDS = ["Arcly", "Nimbus", "Voltra", "Kestrel", "Orbit"]

def make_catalog(n: int, seed: int) -> str:
    rng = random.Random(seed)
    lines = []
    for i in range(n):
        lines.append(f"SKU-{1000 + i} | {rng.choice(BRANDS)} {KINDS[i % len(KINDS)]} {chr(65 + i % 26)}{i // 26 + 1} | "
                     f"${round(rng.uniform(19, 349), 2)} | stock {rng.randint(0, 250)} | warranty {rng.choice([6, 12, 24, 36])} mo | "
                     f"returns {rng.choice([14, 30, 60])} days | works with "
                     + ", ".join(rng.sample(["Windows", "macOS", "Linux", "iPadOS", "ChromeOS", "Android"], 3)))
    return "\n".join(lines)
POLICIES = (
    "Support policies:\n"
    "- Refunds go to the original payment method within 5-7 business days after the return is received.\n"
    "- Items out of stock can be pre-ordered; pre-orders ship within 10 business days.\n"
    "- Warranty claims need the order id and a photo of the fault.\n"
    "- Free shipping on orders over $75; otherwise $6.95.\n"
    "- Answer only from the catalog below. If an item isn't listed, say so.\n"
)
CATALOG = make_catalog(150, seed=42)

def system_prompt(catalog: str) -> str:
    return f"You are the support assistant for an electronics shop (session {RUN}).\n\n{POLICIES}\nCatalog:\n{catalog}"

def count_system(text: str) -> int:
    return client.messages.count_tokens(model=MODEL, system=text, messages=[{"role": "user", "content": "x"}]).input_tokens

FULL = system_prompt(CATALOG)
SHORT = system_prompt("\n".join(CATALOG.split("\n")[:55]))
sizes = {"full prompt tokens": count_system(FULL), "short prompt tokens": count_system(SHORT), "minimum cacheable": MIN_CACHEABLE}
print(sizes)
assert sizes["full prompt tokens"] > MIN_CACHEABLE > sizes["short prompt tokens"]

{'full prompt tokens': 7928, 'short prompt tokens': 2971, 'minimum cacheable': 4096}


## 2. Silent failure #1: too short to cache

We mark a ~3K-token prompt for caching and send it 3 times. On Claude Haiku 4.5 the minimum cacheable prefix is
4,096 tokens, so watch the cache columns: **no error, no warning, just nothing cached.**

In [3]:
def ask(system_text: str, question: str, cached: bool):
    sys_block = [{"type": "text", "text": system_text, **({"cache_control": {"type": "ephemeral"}} if cached else {})}]
    t0 = time.perf_counter(); first = None
    with client.messages.stream(model=MODEL, max_tokens=150, system=sys_block,
                                messages=[{"role": "user", "content": question}]) as s:
        for _ in s.text_stream:
            first = first or time.perf_counter() - t0
        m = s.get_final_message()
    u = m.usage
    return {"input": u.input_tokens, "cache write": u.cache_creation_input_tokens or 0, "cache read": u.cache_read_input_tokens or 0,
            "output": u.output_tokens, "first text (s)": round(first or 0, 3), "cost $": cost(u)}

short_runs = pd.DataFrame([{"call": i + 1, **ask(SHORT, "Which SKUs work with Linux?", cached=True)} for i in range(3)])
short_runs

,call,input,cache write,cache read,output,first text (s),cost $
0,1,2977,0,0,150,0.433,0.003727
1,2,2977,0,0,150,0.415,0.003727
2,3,2977,0,0,150,0.384,0.003727


## 3. Caching done right: 10 questions, cached vs not

Same 8K-token system prompt, same 10 questions, run twice: once with no caching, once with a cache breakpoint on the
system prompt (5-minute TTL). The questions differ, the prefix doesn't.

In [4]:
QUESTIONS = [
    "What's the warranty on SKU-1003?", "Is SKU-1047 in stock?", "What's the return window for SKU-1120?",
    "Which Webcams cost less than $100?", "Does SKU-1088 work with macOS?", "How much is shipping on a $60 order?",
    "What's the cheapest Portable SSD?", "How long do refunds take?", "Can I pre-order SKU-1015?",
    "Which Docking Stations have a 36-month warranty?",
]
uncached = pd.DataFrame([{"q": i + 1, **ask(FULL, q, cached=False)} for i, q in enumerate(QUESTIONS)])
cached = pd.DataFrame([{"q": i + 1, **ask(FULL, q, cached=True)} for i, q in enumerate(QUESTIONS)])

def input_cost(df) -> float:   # input side only: output length varies run to run, the prompt doesn't
    return (df["input"].sum() * PRICE["input"] + df["cache write"].sum() * PRICE["cache_write"]
            + df["cache read"].sum() * PRICE["cache_read"]) / 1e6

def summarise(df, name):
    return {"setup": name, "input tokens (full price)": int(df["input"].sum()), "cache write tokens": int(df["cache write"].sum()),
            "cache read tokens": int(df["cache read"].sum()), "input cost $": round(input_cost(df), 5),
            "output tokens": int(df["output"].sum()), "total cost $": round(df["cost $"].sum(), 5),
            "median first text (s)": round(df["first text (s)"].median(), 3)}

compare = pd.DataFrame([summarise(uncached, "no caching"), summarise(cached, "cached (5-min TTL)")])
saving = 1 - compare.loc[1, "total cost $"] / compare.loc[0, "total cost $"]
input_saving = 1 - compare.loc[1, "input cost $"] / compare.loc[0, "input cost $"]
print(f"input cost saving over 10 questions: {input_saving:.0%} | total cost saving (incl. output): {saving:.0%}")
print("cached run, per call:")
print(cached[["q", "input", "cache write", "cache read", "first text (s)", "cost $"]].to_string(index=False))
compare

input cost saving over 10 questions: 78% | total cost saving (incl. output): 74%
cached run, per call:
 q  input  cache write  cache read  first text (s)   cost $
 1     17         7921           0           0.533 0.010138
 2     16            0        7921           0.467 0.001268
 3     18            0        7921           0.449 0.000965
 4     16            0        7921           0.568 0.001558
 5     18            0        7921           0.446 0.001050
 6     17            0        7921           0.549 0.001054
 7     16            0        7921           0.613 0.001423
 8     13            0        7921           0.450 0.001010
 9     17            0        7921           0.438 0.001559
10     20            0        7921           0.446 0.001562


,setup,input tokens (full price),cache write tokens,cache read tokens,input cost $,output tokens,total cost $,median first text (s)
0,no caching,79378,0,0,0.07938,853,0.08364,0.444
1,cached (5-min TTL),168,7921,71289,0.01720,878,0.02159,0.458


### Does it make responses faster?

At ~8K tokens, time to first text barely moved: reading 8K tokens is already fast. Caching skips re-processing the
prefix, so the effect should grow with prefix size. Same test with a ~50K-token catalog (1,000 products),
5 calls each way. The first cached call is the write; calls 2-5 are reads.

In [5]:
BIG = system_prompt(make_catalog(1000, seed=7))
big_tokens = count_system(BIG)
big_uncached = pd.DataFrame([{"q": i + 1, **ask(BIG, q, cached=False)} for i, q in enumerate(QUESTIONS[:5])])
big_cached = pd.DataFrame([{"q": i + 1, **ask(BIG, q, cached=True)} for i, q in enumerate(QUESTIONS[:5])])
latency = {
    "8K prompt": {"no caching": round(uncached["first text (s)"].median(), 3),
                  "cache reads": round(cached["first text (s)"].iloc[1:].median(), 3)},
    f"{big_tokens // 1000}K prompt": {"no caching": round(big_uncached["first text (s)"].median(), 3),
                                      "cache reads": round(big_cached["first text (s)"].iloc[1:].median(), 3)},
}
big_cost = {"no caching": round(input_cost(big_uncached), 5), "cached": round(input_cost(big_cached), 5)}
print("big prompt tokens:", big_tokens)
print("median seconds to first text:", latency)
print("input cost, 5 calls:", big_cost)
print(big_cached[["q", "input", "cache write", "cache read", "first text (s)"]].to_string(index=False))
pd.DataFrame(latency).T

big prompt tokens: 52106
median seconds to first text: {'8K prompt': {'no caching': np.float64(0.444), 'cache reads': np.float64(0.45)}, '52K prompt': {'no caching': np.float64(0.798), 'cache reads': np.float64(0.556)}}
input cost, 5 calls: {'no caching': np.float64(0.26058), 'cached': np.float64(0.08605)}
 q  input  cache write  cache read  first text (s)
 1     17        52099           0           0.764
 2     16            0       52099           0.718
 3     18            0       52099           0.464
 4     16            0       52099           0.535
 5     18            0       52099           0.576


,no caching,cache reads
8K prompt,0.444,0.450
52K prompt,0.798,0.556


## 4. Silent failure #2: one timestamp breaks it

A common pattern: put the current time in the system prompt. Caching is a **prefix match**, so a value that changes
on every request means every call **writes** a new cache entry (at 1.25× the input price) and **never reads** one.

In [6]:
def with_timestamp() -> str:
    return f"Current time: {datetime.now().isoformat(timespec='microseconds')}\n" + FULL

stamped = pd.DataFrame([{"q": i + 1, **ask(with_timestamp(), q, cached=True)} for i, q in enumerate(QUESTIONS[:5])])
stamp_cost = stamped["cost $"].sum()
same5_uncached = uncached["cost $"].iloc[:5].sum()
same5_cached = cached["cost $"].iloc[:5].sum()
print(f"5 questions: no caching ${same5_uncached:.5f} | caching done right ${same5_cached:.5f} | timestamp in prompt ${stamp_cost:.5f}")
stamped

5 questions: no caching $0.04158 | caching done right $0.01498 | timestamp in prompt $0.05183


,q,input,cache write,cache read,output,first text (s),cost $
0,1,17,7941,0,45,0.435,0.010168
1,2,16,7941,0,94,0.574,0.010412
2,3,18,7941,0,62,0.450,0.010254
3,4,16,7941,0,150,0.431,0.010692
4,5,18,7941,0,72,0.607,0.010304


## 5. When does caching pay off?

With the 5-minute TTL, the first request pays 1.25× for the cached part and every later request pays 0.1×.
Here is the cost of the cached prefix alone (in units of one uncached read of it) for *n* requests within the TTL.

In [7]:
rows = []
for n in [1, 2, 3, 5, 10, 100]:
    no_cache = n * 1.0
    with_cache = 1.25 + 0.10 * (n - 1)
    rows.append({"requests": n, "no caching": round(no_cache, 2), "with caching": round(with_cache, 2),
                 "saving %": round((1 - with_cache / no_cache) * 100)})
breakeven = pd.DataFrame(rows)
breakeven

,requests,no caching,with caching,saving %
0,1,1.0,1.25,-25
1,2,2.0,1.35,32
2,3,3.0,1.45,52
3,5,5.0,1.65,67
4,10,10.0,2.15,78
5,100,100.0,11.15,89


## 6. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [8]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "sizes": sizes,
    "short_runs": short_runs.to_dict("records"),
    "uncached": uncached.to_dict("records"), "cached": cached.to_dict("records"),
    "compare": compare.to_dict("records"), "saving_pct": round(saving * 100), "input_saving_pct": round(input_saving * 100),
    "big_tokens": big_tokens, "big_uncached": big_uncached.to_dict("records"), "big_cached": big_cached.to_dict("records"),
    "latency": latency, "big_input_cost": big_cost,
    "stamped": stamped.to_dict("records"), "stamped_input_cost": round(input_cost(stamped), 5),
    "five_q_input": {"no caching": round(input_cost(uncached.iloc[:5]), 5), "cached right": round(input_cost(cached.iloc[:5]), 5),
                     "timestamp": round(input_cost(stamped), 5)},
    "five_q": {"no caching": round(same5_uncached, 5), "cached right": round(same5_cached, 5), "timestamp": round(stamp_cost, 5)},
    "breakeven": breakeven.to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps({k: f[k] for k in ("sizes", "compare", "saving_pct", "input_saving_pct", "big_tokens", "latency",
                                    "big_input_cost", "five_q", "five_q_input")}, indent=1, default=str))

{
 "sizes": {
  "full prompt tokens": 7928,
  "short prompt tokens": 2971,
  "minimum cacheable": 4096
 },
 "compare": [
  {
   "setup": "no caching",
   "input tokens (full price)": 79378,
   "cache write tokens": 0,
   "cache read tokens": 0,
   "input cost $": 0.07938,
   "output tokens": 853,
   "total cost $": 0.08364,
   "median first text (s)": 0.444
  },
  {
   "setup": "cached (5-min TTL)",
   "input tokens (full price)": 168,
   "cache write tokens": 7921,
   "cache read tokens": 71289,
   "input cost $": 0.0172,
   "output tokens": 878,
   "total cost $": 0.02159,
   "median first text (s)": 0.458
  }
 ],
 "saving_pct": 74,
 "input_saving_pct": 78,
 "big_tokens": 52106,
 "latency": {
  "8K prompt": {
   "no caching": 0.444,
   "cache reads": 0.45
  },
  "52K prompt": {
   "no caching": 0.798,
   "cache reads": 0.556
  }
 },
 "big_input_cost": {
  "no caching": 0.26058,
  "cached": 0.08605
 },
 "five_q": {
  "no caching": 0.04158,
  "cached right": 0.01498,
  "timestamp": 0.0